In [109]:
##ADD ALL IMPORTS HERE FOR ALL CELLS##

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [96]:
PM25 = pd.read_csv("PM2.5.csv", skiprows=4, encoding="utf-8-sig")
PrimarySchoolGross = pd.read_csv('Primary_School_Enrollment_%Gross.csv', skiprows=4, encoding="utf-8-sig")
LifeExpectancy = pd.read_csv('Life_Expectancy.csv', skiprows=4, encoding="utf-8-sig")
GDPperCap = pd.read_csv('GDP_PCAP.csv', skiprows=4, encoding="utf-8-sig")
ForestArea = pd.read_csv('Forest_Area.csv', skiprows=4, encoding="utf-8-sig")
EnergyUse = pd.read_csv('EnergyUseData.csv', skiprows=4, encoding="utf-8-sig")
ExportofGnS = pd.read_csv('ExportofGoodandService.csv', skiprows=4, encoding="utf-8-sig")
RenewableElectricty = pd.read_csv('RenewableElectricyOutput.csv', skiprows=4, encoding="utf-8-sig")
WomenParliament = pd.read_csv('ProportionWomeninParliamnet.csv', skiprows=4, encoding="utf-8-sig")
PopulationDensity = pd.read_csv('PopulationDensity.csv', skiprows=4, encoding="utf-8-sig")
LivestockProduction = pd.read_csv('LivestockProductionIndex.csv', skiprows=4, encoding="utf-8-sig")

Datasets = {
    'PM25': PM25,
    'PrimarySchoolGross': PrimarySchoolGross,
    'LifeExpectancy': LifeExpectancy,
    'GDPperCap': GDPperCap,
    'ForestArea': ForestArea,
    'EnergyUse': EnergyUse,
    'ExportofGnS': ExportofGnS,
    'RenewableElectricty': RenewableElectricty,
    'WomenParliament': WomenParliament,
    'PopulationDensity': PopulationDensity,
    'LivestockProduction': LivestockProduction
}

new_df = {}
for name, df in Datasets.items():
    temp = df.melt(
        id_vars=['Country Name', 'Country Code', 'Indicator Name'],
        var_name='Year',
        value_name=name
    )
    temp = temp[temp['Year'].between('2000','2020')]
    temp['Year'] = temp['Year'].astype(int)
    new_df[name] = temp[['Country Name', 'Country Code', 'Year', name]]

fullset = new_df['PM25']
for name, df in new_df.items():
    if name != 'PM25':
        fullset = fullset.merge(df, on=['Country Name', 'Country Code', 'Year'], how='left')

sovereign_countries_iso3 = [
    'AFG','ALB','DZA','AND','AGO','ATG','ARG','ARM','AUS','AUT','AZE','BHS','BHR','BGD',
    'BRB','BLR','BEL','BLZ','BEN','BTN','BOL','BIH','BWA','BRA','BRN','BGR','BFA','BDI',
    'CPV','KHM','CMR','CAN','CAF','TCD','CHL','CHN','COL','COM','COD','COG','CRI','HRV',
    'CUB','CYP','CZE','DNK','DJI','DMA','DOM','ECU','EGY','SLV','GNQ','ERI','EST','SWZ',
    'ETH','FJI','FIN','FRA','GAB','GMB','GEO','DEU','GHA','GRC','GRD','GTM','GIN','GNB',
    'GUY','HTI','HND','HUN','ISL','IND','IDN','IRN','IRQ','IRL','ISR','ITA','JAM','JPN',
    'JOR','KAZ','KEN','KIR','KWT','KGZ','LAO','LVA','LBN','LSO','LBR','LBY','LIE','LTU',
    'LUX','MDG','MWI','MYS','MDV','MLI','MLT','MHL','MRT','MUS','MEX','FSM','MDA','MCO',
    'MNG','MNE','MAR','MOZ','MMR','NAM','NRU','NPL','NLD','NZL','NIC','NER','NGA','PRK',
    'MKD','NOR','OMN','PAK','PLW','PAN','PNG','PRY','PER','PHL','POL','PRT','QAT','ROU',
    'RUS','RWA','KNA','LCA','VCT','WSM','SMR','STP','SAU','SEN','SRB','SYC','SLE','SGP',
    'SVK','SVN','SLB','SOM','ZAF','KOR','SSD','ESP','LKA','SDN','SUR','SWE','CHE','SYR',
    'TJK','TZA','THA','TLS','TGO','TON','TTO','TUN','TUR','TKM','TUV','UGA','UKR','ARE',
    'GBR','USA','URY','UZB','VUT','VEN','VNM','YEM','ZMB','ZWE'
]

fullset = fullset[fullset['Country Code'].isin(sovereign_countries_iso3)]
fullset.to_csv('fullset.csv', index=False)
print(len(fullset['Country Name'].unique()))
fullset.to_csv('fullset.csv', index = False)

192


In [107]:
fullset = fullset.dropna(subset=['PM25']).sort_values(['Country Code','Year'])

feature_cols = ['PrimarySchoolGross','GDPperCap','ForestArea','EnergyUse',
                'ExportofGnS','RenewableElectricty','WomenParliament',
                'PopulationDensity','LivestockProduction','LifeExpectancy']

fullset[feature_cols] = fullset[feature_cols].apply(pd.to_numeric, errors='coerce')

fullset[feature_cols] = (
    fullset.groupby('Country Code')[feature_cols]
           .transform(lambda x: x.interpolate().ffill().bfill())
)

fullset = fullset.dropna(subset=feature_cols)

fullset.isna().sum()

Country Name           0
Country Code           0
Year                   0
PM25                   0
PrimarySchoolGross     0
LifeExpectancy         0
GDPperCap              0
ForestArea             0
EnergyUse              0
ExportofGnS            0
RenewableElectricty    0
WomenParliament        0
PopulationDensity      0
LivestockProduction    0
dtype: int64

In [125]:
env_vars = ['ForestArea','RenewableElectricty','EnergyUse']
econ_vars = ['GDPperCap','ExportofGnS','LivestockProduction']
soc_vars = ['LifeExpectancy','PrimarySchoolGross','WomenParliament','PopulationDensity']

scaler = StandardScaler()

env_index = PCA(n_components=1).fit_transform(scaler.fit_transform(fullset[env_vars]))
econ_index = PCA(n_components=1).fit_transform(scaler.fit_transform(fullset[econ_vars]))
soc_index = PCA(n_components=1).fit_transform(scaler.fit_transform(fullset[soc_vars]))

fullset['Env_Index'] = env_index
fullset['Econ_Index'] = econ_index
fullset['Soc_Index'] = soc_index

fullset['USI'] = (
    fullset['Env_Index'] +
    fullset['Econ_Index'] +
    fullset['Soc_Index']
) / 3

CountryTable = ['United States','China','India','United Kingdom','Germany']

USItable = fullset[(fullset['Year'] == 2015) &
                  (fullset['Country Name'].isin(CountryTable))]

USItable[['Country Name','Year','Env_Index','Econ_Index','Soc_Index','USI']]

,Country Name,Year,Env_Index,Econ_Index,Soc_Index,USI
4030,China,2015,-0.421575,-0.576706,0.615004,-0.127759
4045,Germany,2015,-0.333575,1.229724,1.665536,0.853895
4071,United Kingdom,2015,-0.739794,0.984831,1.426263,0.557100
4099,India,2015,-0.271623,-0.861883,0.070094,-0.354471
4241,United States,2015,-1.161906,0.999550,0.575899,0.137848
